# Guião 6

In [2]:
import random, os, struct
from abc import ABC, abstractmethod
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives import hashes, hmac


class Cipher_(ABC):
    @abstractmethod
    def keygen(self) -> bytes:
        pass

    @abstractmethod
    def enc(self, key: bytes, text: bytes) -> bytes:
        pass

    @abstractmethod
    def dec(self, key: bytes, ciphertext: bytes) -> bytes:
        pass

class INDCPA_Adv(ABC):
    @abstractmethod
    def choose(self, oracle: callable):
        pass

    @abstractmethod
    def guess(self, oracle: callable, ciphertext: bytes) -> int:
        pass

---
# `indcpa.py` — Jogo IND-CPA

Animação do jogo IND-CPA com:
1. **Cifra Identidade** — adversário com estratégia óbvia → 100% sucesso
2. **ChaCha20** — adversário com estratégia aleatória → ~50% sucesso

In [3]:
def IND_CPA(C: Cipher_, A: INDCPA_Adv) -> bool:
    k = C.keygen()
    enc_oracle = lambda ptxt: C.enc(k, ptxt)
    m0, m1 = A.choose(enc_oracle)
    assert len(m0) == len(m1), "As mensagens devem ter o mesmo tamanho"
    b = random.randint(0, 1)
    c = C.enc(k, m1 if b else m0)
    b_prime = A.guess(enc_oracle, c)
    return b == b_prime

In [4]:
#1. Cifra Identidade

class IdentityCipher(Cipher_):
    def keygen(self) -> bytes:
        return b''

    def enc(self, key: bytes, text: bytes) -> bytes:
        return text

    def dec(self, key: bytes, ciphertext: bytes) -> bytes:
        return ciphertext


class IdentityAdversary(INDCPA_Adv):

    def choose(self, oracle):
        self.m0 = b'message_A'
        self.m1 = b'message_B'
        return self.m0, self.m1

    def guess(self, oracle, ciphertext):
        return 1 if ciphertext == self.m1 else 0


C = IdentityCipher()
A = IdentityAdversary()
limite = 1000
sucesso = sum(IND_CPA(C, A) for _ in range(limite))
print(f"[Cifra Identidade] Sucesso: {sucesso/limite*100:.1f}%  (esperado: 100%)")

[Cifra Identidade] Sucesso: 100.0%  (esperado: 100%)


In [5]:
# 2. ChaCha20


class ChaCha20Cipher(Cipher_):
    def keygen(self) -> bytes:
        return os.urandom(32)

    def enc(self, key: bytes, text: bytes) -> bytes:
        nonce = os.urandom(8)
        counter = 0
        full_nonce = struct.pack("<Q", counter) + nonce  # 16 bytes: 8 contador + 8 nonce
        algorithm = algorithms.ChaCha20(key, full_nonce)
        cipher = Cipher(algorithm, mode=None)
        ct = cipher.encryptor().update(text)
        return full_nonce + ct

    def dec(self, key: bytes, ciphertext: bytes) -> bytes:
        full_nonce = ciphertext[:16]
        ct = ciphertext[16:]
        algorithm = algorithms.ChaCha20(key, full_nonce)
        cipher = Cipher(algorithm, mode=None)
        return cipher.decryptor().update(ct)


class RandomAdversary(INDCPA_Adv):

    def choose(self, oracle):
        self.m0 = b'message_A'
        self.m1 = b'message_B'
        return self.m0, self.m1

    def guess(self, oracle, ciphertext):
        return random.randint(0, 1)  


C = ChaCha20Cipher()
A = RandomAdversary()
limite = 100_000
sucesso = sum(IND_CPA(C, A) for _ in range(limite))
print(f"[ChaCha20]        Sucesso: {sucesso/limite*100:.2f}%  (esperado: ~50%)")
print(f"                  Vantagem: {abs(sucesso/limite - 0.5)*200:.2f}%  (esperado: ~0%, negligenciável)")

[ChaCha20]        Sucesso: 50.01%  (esperado: ~50%)
                  Vantagem: 0.01%  (esperado: ~0%, negligenciável)


---
# `indcpa_attck.py` — Ataque IND-CPA ao modo ECB

In [6]:
# Cifra AES-ECB

class AES_ECB_Cipher(Cipher_):
    def keygen(self) -> bytes:
        return os.urandom(32)

    def enc(self, key: bytes, text: bytes) -> bytes:
        cipher = Cipher(algorithms.AES(key), mode=modes.ECB())
        encryptor = cipher.encryptor()
        return encryptor.update(text) + encryptor.finalize()

    def dec(self, key: bytes, ciphertext: bytes) -> bytes:
        cipher = Cipher(algorithms.AES(key), mode=modes.ECB())
        decryptor = cipher.decryptor()
        return decryptor.update(ciphertext) + decryptor.finalize()


# Adversário IND-ECB

class ECB_Adversary(INDCPA_Adv):

    def choose(self, oracle):
        self.m0 = b'A' * 16 + b'A' * 16   # dois blocos iguais
        self.m1 = b'A' * 16 + b'B' * 16   # dois blocos diferentes
        return self.m0, self.m1

    def guess(self, oracle, ciphertext):
        bloco1 = ciphertext[0:16]
        bloco2 = ciphertext[16:32]
        # Se os blocos do CT são iguais → foi cifrado m0 (b=0)
        return 0 if bloco1 == bloco2 else 1


C = AES_ECB_Cipher()
A = ECB_Adversary()
limite = 1000
sucesso = sum(IND_CPA(C, A) for _ in range(limite))
print(f"[AES-ECB IND-CPA] Sucesso: {sucesso/limite*100:.1f}%  (esperado: 100%)")

[AES-ECB IND-CPA] Sucesso: 100.0%  (esperado: 100%)


---
# Outros Modelos

## IND-CCA — Insegurança do AES-CTR

In [7]:
# Cifra AES-CTR

class AES_CTR_Cipher(Cipher_):
    def keygen(self) -> bytes:
        return os.urandom(32)

    def enc(self, key: bytes, text: bytes) -> bytes:
        nonce = os.urandom(16)
        cipher = Cipher(algorithms.AES(key), mode=modes.CTR(nonce))
        encryptor = cipher.encryptor()
        ct = encryptor.update(text) + encryptor.finalize()
        return nonce + ct

    def dec(self, key: bytes, ciphertext: bytes) -> bytes:
        nonce = ciphertext[:16]
        ct = ciphertext[16:]
        cipher = Cipher(algorithms.AES(key), mode=modes.CTR(nonce))
        decryptor = cipher.decryptor()
        return decryptor.update(ct) + decryptor.finalize()


#Jogo IND-CCA (adversário tem oráculo de decifra)

def IND_CCA(C: Cipher_, A) -> bool:
    k = C.keygen()
    enc_oracle = lambda ptxt: C.enc(k, ptxt)
    dec_oracle = lambda ctxt: C.dec(k, ctxt)
    m0, m1 = A.choose(enc_oracle)
    assert len(m0) == len(m1)
    b = random.randint(0, 1)
    c = C.enc(k, m1 if b else m0)
    b_prime = A.guess(dec_oracle, enc_oracle, c)
    return b == b_prime


#Adversário IND-CCA

class CTR_CCA_Adversary:
    def choose(self, enc_oracle):
        self.m0 = b'Acesso Negado!!'
        self.m1 = b'Acesso Aceite!!'
        return self.m0, self.m1

    def guess(self, dec_oracle, enc_oracle, ciphertext):
        delta = 0xFF
        c_mod = bytearray(ciphertext)
        c_mod[-1] ^= delta            

        m_mod = dec_oracle(bytes(c_mod))

        m_orig = bytearray(m_mod)
        m_orig[-1] ^= delta

        return 0 if bytes(m_orig) == self.m0 else 1


C = AES_CTR_Cipher()
A = CTR_CCA_Adversary()
limite = 1000
sucesso = sum(IND_CCA(C, A) for _ in range(limite))
print(f"[AES-CTR IND-CCA] Sucesso: {sucesso/limite*100:.1f}%  (esperado: 100%)")

[AES-CTR IND-CCA] Sucesso: 100.0%  (esperado: 100%)


## INT-PTXT — Hash simples (sem chave) como MAC

In [8]:
# "Cifra" com hash simples sem chave


class SimpleHashCipher:
    def keygen(self) -> bytes:
        return os.urandom(32)

    def enc(self, text: bytes) -> bytes:
        h = hashes.Hash(hashes.SHA256())
        h.update(text)
        digest = h.finalize()
        return digest + text

    def dec(self, ciphertext: bytes) -> bytes:
        digest_recebido = ciphertext[:32]
        text = ciphertext[32:]
        h = hashes.Hash(hashes.SHA256())
        h.update(text)
        if digest_recebido == h.finalize():
            return text
        return None


#INT-PTXT 


def IND_simples_hash(C: SimpleHashCipher, A) -> bool:
    k = C.keygen()
    enc_oracle = lambda ptxt: C.enc(ptxt)   # chave ignorada
    m0, m1 = A.choose(enc_oracle)
    assert len(m0) == len(m1)
    b = random.randint(0, 1)
    c = C.enc(m1 if b else m0)
    b_prime = A.guess(enc_oracle, c)
    return b == b_prime


# Adversário INT-PTXT 

class SimpleHashAdversary:

    def choose(self, enc_oracle):
        self.m0 = b'Acesso Negado!!'
        self.m1 = b'Acesso Aceite!!'
        return self.m0, self.m1

    def guess(self, enc_oracle, ciphertext):
        m0_enc = enc_oracle(self.m0)
        return 0 if m0_enc[:32] == ciphertext[:32] else 1


C = SimpleHashCipher()
A = SimpleHashAdversary()
limite = 1000
sucesso = sum(IND_simples_hash(C, A) for _ in range(limite))
print(f"[Hash simples INT-PTXT] Sucesso: {sucesso/limite*100:.1f}%  (esperado: 100%)")

[Hash simples INT-PTXT] Sucesso: 100.0%  (esperado: 100%)
